# Exercise 12 - Custom implementation for LayerNormalization

In [1]:
import tensorflow as tf
from tensorflow import keras

In [6]:
class LayerNormalization(keras.layers.Layer):
    def __init__(self, epsilon=0.001, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.alpha = self.add_weight(name="alpha", shape=input_shape[-1:], dtype=tf.float32, initializer="ones", trainable=True)
        self.beta = self.add_weight(name="beta", shape=input_shape[-1:], dtype=tf.float32, initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        mean, variance = tf.nn.moments(inputs, axes=-1, keepdims=True)
        normalized = (inputs - mean) / tf.sqrt(variance + self.epsilon)
        return self.alpha * normalized + self.beta

In [7]:
tf.random.set_seed(42)
X = tf.random.normal(shape=(2, 3, 4))

custom_layer = LayerNormalization()
keras_layer = keras.layers.LayerNormalization(epsilon=0.001, axis=-1)

custom_output = custom_layer(X)
keras_output = keras_layer(X)

print("Custom output:\n", custom_output.numpy())
print("\nKeras output:\n", keras_output.numpy())

print("\nMaximum absolute difference:", tf.reduce_max(tf.abs(custom_output - keras_output)).numpy())
tf.debugging.assert_near(custom_output, keras_output, atol=1e-5)

Custom output:
 [[[ 0.96849823 -0.58752805  0.9578133  -1.3387837 ]
  [-1.4521675  -0.16931194  0.28904676  1.3324326 ]
  [ 1.4287091   0.20376745 -1.3555722  -0.27690446]]

 [[-0.02667039 -1.3738577  -0.05036796  1.4508961 ]
  [ 0.10058279 -1.6465416   0.9754772   0.57048154]
  [ 1.2099249   0.31996492 -1.5562718   0.02638202]]]

Keras output:
 [[[ 0.96849823 -0.5875281   0.95781326 -1.3387836 ]
  [-1.4521673  -0.16931194  0.28904676  1.3324325 ]
  [ 1.4287093   0.20376745 -1.3555722  -0.27690446]]

 [[-0.0266704  -1.3738577  -0.05036795  1.450896  ]
  [ 0.10058281 -1.6465416   0.9754772   0.5704815 ]
  [ 1.2099249   0.319965   -1.5562719   0.02638209]]]

Maximum absolute difference: 2.3841858e-07
